# Hypervector Storage

`InMemory` provides ephemeral storage for notebooks — data disappears when the
Python process exits. `Fjall` provides persistent disk-backed storage. Both
implement the same `Memory` trait as Go's `badger.Memory`, with `mem_set` /
`mem_get` exposing the producer/selector interface. Convenience methods `put` /
`get` / `delete` / `names` wrap the common patterns.

Use `export(source, dest)` to persist an InMemory session to Fjall.

In [ ]:
# Install / upgrade kongming-rs-hv (required for Google Colab).
# After this cell finishes, restart the runtime before running any other cell:
#   Runtime menu → Restart session  (or Ctrl+M .)
import subprocess, sys
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--upgrade', 'kongming-rs-hv'],
    capture_output=True, text=True
)
print(result.stdout[-800:] if result.stdout else '')
if result.returncode != 0:
    print('ERROR:', result.stderr[-800:])
print("Restart the kernel before importing (Kernel → Restart Kernel).")

In [ ]:
from kongming_rs import api, hv, memory

## Create a store

In [ ]:
storage = memory.InMemory(api.MODEL_64K_8BIT)
storage.name, storage.store_id, len(storage)

## Memory-trait API: `mem_set(producer)` / `mem_get(selector)`

These mirror Go's `Memory.Set(ChunkProducer)` / `Memory.Get(ChunkSelector)`.

Producers and selectors are first-class objects — construct them with factory
functions like `write_chunk_producer()`, `by_item_key()`, etc.

In [ ]:
words = hv.Domain.from_name("words")
hello = hv.Sparkle.from_word(api.MODEL_64K_8BIT, words, "hello")

# mem_set takes a Producer — here a terminal producer that creates a sparkle-only chunk
prod = memory.terminal("words", "world")
prod

In [ ]:
returned = storage.mem_set(prod)
type(returned).__name__

In [ ]:
# mem_get takes a Selector — by_item_key selects a single item
results = storage.mem_get(memory.by_item_key("words", "world"))
len(results), type(results[0]).__name__

In [ ]:
# new_learner producer creates a fresh Learner via the Memory trait
lr_via_trait = storage.mem_set(memory.new_learner("words", "my_learner"))
type(lr_via_trait).__name__

## Convenience API: `put` / `get` / `names`

`put(hv)` extracts domain and pod from the HV itself — every type (Sparkle, Set,
Learner, etc.) carries its own domain/pod. It also guards against protected
domains (prefix `~`) and maintains a name registry so `names(domain)` can return
human-readable strings.

In [ ]:
fruits = hv.Domain.from_name("fruits")
apple  = hv.Sparkle.from_word(api.MODEL_64K_8BIT, fruits, "apple")
banana = hv.Sparkle.from_word(api.MODEL_64K_8BIT, fruits, "banana")
cherry = hv.Sparkle.from_word(api.MODEL_64K_8BIT, fruits, "cherry")

storage.put(apple,  note="Granny Smith")
storage.put(banana, note="Cavendish")
storage.put(cherry)

len(storage), storage.names("fruits")

## Retrieve and verify

`get` returns the exact HyperBinary type that was stored.

In [ ]:
got_apple = storage.get("fruits", "apple")
hv.equal(got_apple, apple), type(got_apple).__name__

In [ ]:
storage.note("fruits", "apple"), storage.note("fruits", "cherry")

## Store composites

Sets and Octopuses round-trip through the store. Probe with `overlap` and `release`.

In [ ]:
fruit_set = hv.Set(fruits, hv.Pod.from_word("fruit_set"), apple, banana, cherry)
storage.put(fruit_set)

got_set = storage.get("fruits", "fruit_set")
type(got_set).__name__

In [ ]:
# Probe the set — unmasked() strips the marker, revealing bundled members
unmasked = got_set.unmasked()
hv.overlap(unmasked, apple), hv.overlap(unmasked, banana), hv.overlap(unmasked, cherry)

In [ ]:
# Store an Octopus (key-value structure)
octo = hv.Octopus(fruits, hv.Pod.from_word("color_map"), ["fruit", "color"], apple, cherry)
storage.put(octo)

got_octo = storage.get("fruits", "color_map")
fruit_key = hv.Sparkle.from_word(api.MODEL_64K_8BIT, hv.d0(), "fruit")
color_key = hv.Sparkle.from_word(api.MODEL_64K_8BIT, hv.d0(), "color")
hv.overlap(hv.release(got_octo, fruit_key), apple), hv.overlap(hv.release(got_octo, color_key), cherry)

## Near-neighbor search

The associative index enables content-based retrieval. Given a composite
(Set, Sequence, Octopus), attractors strip the structural encoding and
NNS finds the stored items that best match.

In [ ]:
# Add a non-member to show NNS selectivity
grape = hv.Sparkle.from_word(api.MODEL_64K_8BIT, fruits, "grape")
storage.put(grape)

# NNS: "which stored items are members of fruit_set?"
#
# set_members() is an attractor — it strips the Set marker to expose the
# bundled content. nns() then uses that unmasked content to search the
# associative index for stored items that overlap with it.
# Without nns(), set_members() alone would just return the set itself
# with modified code — it wouldn't find the individual members.
results = storage.mem_get(memory.nns(memory.set_members(memory.by_item_key(fruits, "fruit_set"))))
print(f"Set members found: {len(results)}")
for r in results:
    print(f"  apple={hv.overlap(r, apple):3d}  banana={hv.overlap(r, banana):3d}  "
          f"cherry={hv.overlap(r, cherry):3d}  grape={hv.overlap(r, grape):3d}")

In [ ]:
# NNS with tentacle attractor: "what value is stored under key 'fruit' in color_map?"
#
# tentacle() strips the key binding from the Octopus, exposing the value.
# nns() then finds stored items matching that value via the index.
results = storage.mem_get(memory.nns(memory.tentacle(memory.by_item_key(fruits, "color_map"), "fruit")))
print(f"Tentacle 'fruit' found: {len(results)}")
for r in results:
    print(f"  apple={hv.overlap(r, apple):3d}  cherry={hv.overlap(r, cherry):3d}")

## Learner workflow

Store a Learner, retrieve it, bundle observations, and put the updated version back.

In [ ]:
learner = hv.Learner(api.MODEL_64K_8BIT, fruits, hv.Pod.from_word("fruit_learner"))
storage.put(learner)

# Retrieve, train, and store the updated learner
lr = storage.get("fruits", "fruit_learner")
lr.bundle(apple)
lr.bundle(banana)
lr.bundle(cherry)
storage.put(lr, note="3 observations")

# Verify the updated learner persists
lr2 = storage.get("fruits", "fruit_learner")
lr2.weight(apple), lr2.weight(banana), hv.overlap(lr2, apple)

## Cross-domain isolation

Different domains are independent namespaces.

In [ ]:
colors = hv.Domain.from_name("colors")
red = hv.Sparkle.from_word(api.MODEL_64K_8BIT, colors, "red")
storage.put(red)

storage.names("fruits"), storage.names("colors")

## Export to persistent storage

`export(source, dest)` copies all entries (items + index) from one store
to another via the Substrate scan/write interface.

In [ ]:
import tempfile, os
tmpdir = tempfile.mkdtemp()
path = os.path.join(tmpdir, "my_store")

dest = memory.Fjall(api.MODEL_64K_8BIT, path)
count = memory.export(storage, dest)
print(f"Exported {count} entries to disk")

# Verify data survived the export
got = dest.get(fruits, "apple")
hv.equal(got, apple)

In [ ]:
# Reopen from disk — data persists
del dest
reopened = memory.Fjall(api.MODEL_64K_8BIT, path)
got2 = reopened.get(fruits, "banana")
print(f"Reopened from disk: equal={hv.equal(got2, banana)}")

# Cleanup
import shutil
shutil.rmtree(tmpdir)